# JN0c · What a function is

*On-ramp 3 of 8.*

We're about to ask one question — *does this permit create new housing?* — about **thirty thousand** permits. How do we make sure we ask it the *exact same way* every single time? With a **function**.

### Running the cells

To run a cell, click it and press **Shift + Return**, or click the **run (▸) button** on the cell. The simplest way through any notebook here is to start at the top and run each cell in order, reading the output that appears beneath it.

Some of the computational cells may look complex right now — that's expected, and it's fine. **You don't need to understand every line yet;** the ideas become clear as you go. Run them, watch what they produce, and keep moving.

💡 Tip: the **Next** link opens the following notebook in a new tab. If Colab says you have too many sessions, just close the previous tab and continue.

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← Previous: [JN0b · What a computational notebook is](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0b_notebook.ipynb)  |  Next: [JN0d · What a pandas DataFrame is](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0d_dataframe.ipynb) →

## (run first) Colab setup

Fetches the data + shared modules from R2. **No-op if you already have the repo locally.** On Colab it recreates the minimal layout so the cells below find everything.

In [1]:
# === COLAB BOOTSTRAP - fetch curriculum data + modules from R2 (NO-OP if the repo is local) ===
from pathlib import Path
import sys, urllib.request, urllib.parse, tarfile, subprocess

R2 = 'https://pub-2cee87f70da64080ab70ee0a34b55099.r2.dev/curriculum'
USE_CLEAN = False   # False: raw .xlsx path (JN1's messy-data lesson).  True (skip-ingest): permits_clean.*

_here = Path.cwd()
_have_repo = (_here/'scripts'/'build_v2').exists() or any((p/'scripts'/'build_v2').exists() for p in _here.parents)

def _get(url):
    # r2.dev sits behind Cloudflare, which 403s the default 'Python-urllib' User-Agent; send a browser UA.
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

if _have_repo:
    print('local repo detected - no fetch needed')
else:
    try:
        import pyarrow  # the parquet / USE_CLEAN path needs it; Colab has pandas, maybe not pyarrow
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)
    def _fetch(url, dest):
        dest = Path(dest)
        if dest.exists():
            return                                   # cached: re-runs don't re-download
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(_get(url)); print('fetched', dest.name)
    # 1) shared modules -> ./scripts/...  (the config-cell repo-root walk then finds scripts/build_v2)
    if not (_here/'scripts'/'build_v2').exists():
        Path('modules.tgz').write_bytes(_get(f'{R2}/curriculum_modules.tar.gz'))
        _tar = tarfile.open('modules.tgz')
        try: _tar.extractall(_here, filter='data')      # py3.12+: safe extract, no deprecation warning
        except TypeError: _tar.extractall(_here)         # older python has no filter arg
        _tar.close(); Path('modules.tgz').unlink(missing_ok=True)   # tidy: drop the intermediate tarball
        print('extracted modules -> ./scripts/')
    # 2) data -> the SAME relative paths the notebooks use (raw .xlsx AND clean exports, both fetched)
    for rel in ['data/raw/cpra-downloads/BP_Annual Permit Report-2018-2022.xlsx',
                'data/raw/cpra-downloads/BP_Annual Permit Report-2023-2025.xlsx',
                'databases/hcd_apr_mirror_2026-06-17_fresh.db',
                'databases/hcd_apr_mirror.db',
                'data/processed/permits_clean.csv',
                'data/processed/permits_clean.parquet',
                'data/processed/permits_clean_README.md']:
        _fetch(f"{R2}/data/{urllib.parse.quote(rel.split('/')[-1])}", _here/rel)   # quote -> %20 for the spaced .xlsx names
    print('curriculum bundle ready (fetched from R2)')


local repo detected - no fetch needed


In [2]:
def md(t):
    from IPython.display import Markdown, display
    display(Markdown(t))

## Point the notebook at the data

Finds the repo root, locates the permit feed, and puts the project's real shared code on the path. The two knobs near the top are all a student changes to run this on another city.

In [3]:
# === CONFIG — point this at YOUR city's permit data (this notebook is clonable) ===
from pathlib import Path
import sys, glob

# walk up to the repo root (where scripts/build_v2 lives) so the notebook runs from anywhere
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts' / 'build_v2').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# --- the two knobs a student changes for another city ---
PERMIT_GLOB   = str(REPO_ROOT / 'data/raw/cpra-downloads/BP_Annual Permit Report-*.xlsx')
HEADER_ROW    = 7        # 0-indexed: Berkeley's CPRA export puts the column names on row 8
EXPECTED_UNIQUE = 30764  # the known unique-permit total for YOUR feed (Berkeley = 30,764)

# import the REAL shared modules the pipeline uses (we demonstrate them, never reinvent)
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'build_v2'))
print('repo root :', REPO_ROOT)
print('feed files:', [Path(f).name for f in glob.glob(PERMIT_GLOB)])


repo root : /Users/johngage/berkeley-data
feed files: ['BP_Annual Permit Report-2023-2025.xlsx', 'BP_Annual Permit Report-2018-2022.xlsx']


## A named recipe: inputs in, answer out

A **function** is a named recipe. You hand it **inputs** (its *arguments*); it hands back a **return value**. Define it once, use it everywhere — and because it's one definition, the rule can never quietly *drift* from row to row. Here's a tiny one: *is this permit for New construction?*

In [4]:
def is_new(work_type):                 # a tiny named recipe: is this permit for New construction?
    return work_type == 'New'

print(is_new('New'), is_new('Alteration'))   # -> True False

True False


## Use the *real* function, don't reinvent it

The actual pipeline already defines the careful rules — like `net_units`, *how many new homes a permit creates*. Rather than retype it (and risk a subtly different version), we **import** it: pull the function in from its shared **module**, the single source everyone uses. Let's run the real one on two real permits.

In [5]:
import pandas as pd, glob
from housing_predicates import net_units, is_housing   # the REAL shared functions the pipeline runs on
def _load(p):
    # read one spreadsheet at its real header row, then tidy the column names
    d = pd.read_excel(p, dtype=str, header=HEADER_ROW); d.columns = [str(c).strip() for c in d.columns]; return d
df = pd.concat([_load(f) for f in glob.glob(PERMIT_GLOB)], ignore_index=True)
df = df[df['PermitNumber'].notna()].copy()
a = df[df['PermitNumber'] == 'B2023-02847'].iloc[0]   # 'remove electric fireplace' in a 99-unit building
b = df[df['PermitNumber'] == 'B2019-05574'].iloc[0]   # a brand-new 135-unit building
# ask the real function how many NEW homes each permit creates
ua = net_units(a['Work Type'] == 'New', a['UnitsAdded'], a['NumberUnits'], a['ADU'])
ub = net_units(b['Work Type'] == 'New', b['UnitsAdded'], b['NumberUnits'], b['ADU'])
print('B2023-02847 ->', ua)
print('B2019-05574 ->', ub)

B2023-02847 -> 0
B2019-05574 -> 135.0


In [6]:
md(f'''### One function, two honest answers

`net_units` is the **real** function the pipeline uses — we *imported* it, we didn't re-type it. On **B2023-02847** ("remove electric fireplace" in a 99-unit building) it returns **{ua:.0f}**: the building already existed, so this permit makes **no** new homes — even though the row's `NumberUnits` says 99. On **B2019-05574** (a new 135-unit building — the North half of Logan Park, which you'll meet again in the capstone) it returns **{ub:.0f}**. Same function, opposite answers, because it reads the *meaning* of the columns, not just a number.''')

### One function, two honest answers

`net_units` is the **real** function the pipeline uses — we *imported* it, we didn't re-type it. On **B2023-02847** ("remove electric fireplace" in a 99-unit building) it returns **0**: the building already existed, so this permit makes **no** new homes — even though the row's `NumberUnits` says 99. On **B2019-05574** (a new 135-unit building — the North half of Logan Park, which you'll meet again in the capstone) it returns **135**. Same function, opposite answers, because it reads the *meaning* of the columns, not just a number.

In [7]:
df['isnew'] = df['Work Type'] == 'New'            # mark which permits are New construction
# run the real net_units function on EVERY permit, building a new-homes column
df['new_units'] = [net_units(n, ua_, nu_, adu_)
                   for n, ua_, nu_, adu_ in zip(df['isnew'], df['UnitsAdded'], df['NumberUnits'], df['ADU'])]
naive_total = df['new_units'].sum(); n_zero = int((df['new_units'] == 0).sum())   # add them up; count the zeros
md(f'''### The same function, thirty thousand times

One line ran `net_units` on **every** permit. **{n_zero:,}** return 0 — not new housing at all. Add up the rest and the naive total is **{naive_total:,.0f}** units. Hold onto that number: it's *too big*, because it counts every permit on a building separately and never asks which permits describe the **same** building. Turning permits into buildings is exactly what **JN3** does. A function let us ask one precise question 30,000 times in a blink — next we learn to keep all that data in a shape we can query: the **DataFrame**.''')

### The same function, thirty thousand times

One line ran `net_units` on **every** permit. **29,341** return 0 — not new housing at all. Add up the rest and the naive total is **46,320** units. Hold onto that number: it's *too big*, because it counts every permit on a building separately and never asks which permits describe the **same** building. Turning permits into buildings is exactly what **JN3** does. A function let us ask one precise question 30,000 times in a blink — next we learn to keep all that data in a shape we can query: the **DataFrame**.

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← Previous: [JN0b · What a computational notebook is](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0b_notebook.ipynb)  |  Next: [JN0d · What a pandas DataFrame is](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0d_dataframe.ipynb) →